# Contributors
- Samer Roz

# 05 — Comparative offline evaluation (group baseline)

This notebook builds the **single comparison table** used to judge all four recommenders against the same metric columns (aligned with notebooks `01`–`04`). The assignment briefing is not checked into this folder; from the shared notebooks the intended deliverable is a **standardized offline comparison** on Instacart implicit feedback (purchase / reorder signals), not identical train/test pipelines across every model (memory limits led each teammate to evaluate in their own notebook).

## Protocol
- **Ranking metrics** use **K = 10** where applicable (`Precision@10`, `Recall@10`, `NDCG@10`).
- **Coverage** is the fraction of the training catalog touched by the union of top-K recommendations over evaluated users (definitions match each source notebook).
- **RMSE / MAE** are only meaningful where the teammate defined a comparable target (global reorder prediction for Most Popular; score-vs-binary-relevance on top-K for TF‑IDF). Other rows leave these blank.
- **Context-aware** metrics below are computed **here** with the same heuristic as `04_context_aware.ipynb` whenever `orders.csv` and `order_products__prior.csv` are available. If they are missing from the current folder, the first code cell **downloads** the Instacart dataset from Kaggle via **`kagglehub`** (dataset `yasserh/instacart-online-grocery-basket-analysis-dataset`) and points `DATA_DIR` at the extracted CSVs. You can still place CSVs yourself (e.g. from Google Drive) to skip the download.

## Source of truth for each row
- **Non-personalized:** `01_non_personalized.ipynb` — **Most Popular** row from the final comparison table.
- **Collaborative filtering:** `02_collaborative_filtering.ipynb` — **Item-Based CF** row (`comparison_table_cf`).
- **Content-based:** `03_content_based.ipynb` — **Content-Based TF-IDF** (`evaluation_table`; semantic features = product name + aisle + department).
- **Context-aware:** heuristic from `04_context_aware.ipynb`, evaluated in this notebook when data files are available.

In [ ]:
from __future__ import annotations

# Install dependencies as needed (first run / Colab):
!pip install kagglehub

import gc
import math
from pathlib import Path

import numpy as np
import pandas as pd

# --- Instacart data location -----------------------------------------------
# Primary source: Kaggle dataset (same schema as the original Instacart release).
# https://www.kaggle.com/datasets/yasserh/instacart-online-grocery-basket-analysis-dataset
#
# Optional: load a single table into pandas via Hub (not used by this notebook's
# evaluation code, which reads CSV files from disk):
#
#   import kagglehub
#   from kagglehub import KaggleDatasetAdapter
#   df = kagglehub.load_dataset(
#       KaggleDatasetAdapter.PANDAS,
#       "yasserh/instacart-online-grocery-basket-analysis-dataset",
#       "orders.csv",  # path inside the dataset archive
#   )
#
# We need at least orders.csv + order_products__prior.csv on disk for context-aware eval.

KAGGLE_INSTACART = "yasserh/instacart-online-grocery-basket-analysis-dataset"
_REQUIRED_FOR_CONTEXT = ("orders.csv", "order_products__prior.csv")


def _dir_has_required_csvs(d: Path) -> bool:
    return all((d / f).is_file() for f in _REQUIRED_FOR_CONTEXT)


def _resolve_instacart_root(search_under: Path) -> Path | None:
    """Return a folder containing orders.csv (dataset may unzip one level deep)."""
    if _dir_has_required_csvs(search_under):
        return search_under
    for orders in search_under.rglob("orders.csv"):
        parent = orders.parent
        if _dir_has_required_csvs(parent):
            return parent
    return None


DATA_DIR = Path(".").resolve()
if not _dir_has_required_csvs(DATA_DIR):
    try:
        import kagglehub
    except ImportError as err:
        raise ImportError(
            "No Instacart CSVs in the working directory. Either add orders.csv and "
            "order_products__prior.csv here, or install Kaggle Hub:  pip install kagglehub"
        ) from err

    _cache_path = Path(kagglehub.dataset_download(KAGGLE_INSTACART))
    _resolved = _resolve_instacart_root(_cache_path)
    if _resolved is None:
        raise FileNotFoundError(
            f"Downloaded {KAGGLE_INSTACART} to {_cache_path} but could not find "
            f"{_REQUIRED_FOR_CONTEXT}. Inspect the folder or set DATA_DIR manually."
        )
    DATA_DIR = _resolved
    print("Instacart CSVs from Kaggle Hub →", DATA_DIR)
else:
    print("Instacart CSVs in working directory →", DATA_DIR)

K = 10
RANDOM_SEED = 42

# Reproducible subsample size for context-aware eval (raise if RAM allows)
CONTEXT_EVAL_MAX_USERS = 500

Using Colab cache for faster access to the 'instacart-online-grocery-basket-analysis-dataset' dataset.
Instacart CSVs from Kaggle Hub → /kaggle/input/instacart-online-grocery-basket-analysis-dataset


In [ ]:
# --- Metrics frozen from teammate notebooks (re-run those notebooks after code changes, then paste new values here) ---

METRICS_NON_PERSONALIZED = {
    "Approach": "Non-Personalized (Most Popular)",
    "RMSE": 0.7710,
    "MAE": 0.6359,
    "Precision@10": 0.0743,
    "Recall@10": 0.0733,
    "NDCG": 0.1025,
    "Coverage": 0.0004,
    "Diversity": 0.3556,
    "Serendipity": 0.0000,
    "Context": "Non-personalized popularity baseline",
    "_source": "01_non_personalized.ipynb final comparison_table: Most Popular",
}

METRICS_CF = {
    "Approach": "Collaborative Filtering (Item-Based)",
    "RMSE": np.nan,
    "MAE": np.nan,
    "Precision@10": 0.0378,
    "Recall@10": 0.0557,
    "NDCG": 0.0594,
    "Coverage": 0.2520,
    "Diversity": 0.9700,
    "Serendipity": 0.0054,
    "Context": "Personalized item similarity (top-1500 product universe)",
    "_source": "02_collaborative_filtering.ipynb comparison_table_cf",
}

METRICS_CONTENT_TFIDF = {
    "Approach": "Content-Based (TF-IDF)",
    "RMSE": 0.5185,
    "MAE": 0.5181,
    "Precision@10": 0.0025,
    "Recall@10": 0.0020,
    "NDCG": 0.0018,
    "Coverage": 0.0160,
    "Diversity": 0.4183,
    "Serendipity": 0.8455,
    "Context": "Product name + aisle + department",
    "_source": "03_content_based.ipynb evaluation_table",
}

# Use after you run evaluate_context_aware_block() once with CSVs present; avoids empty row in team exports
FALLBACK_CONTEXT_METRICS = {
    "Approach": "Context-Aware (Heuristic)",
    "RMSE": np.nan,
    "MAE": np.nan,
    "Precision@10": np.nan,
    "Recall@10": np.nan,
    "NDCG": np.nan,
    "Coverage": np.nan,
    "Diversity": np.nan,
    "Serendipity": np.nan,
    "Context": "Temporal (day of week, hour of day, recency)",
    "_source": "placeholder — run evaluation cell with Instacart CSVs in this directory",
}

In [ ]:
def precision_at_k(recommended, relevant, k: int = 10) -> float:
    recommended = recommended[:k]
    if k == 0:
        return 0.0
    return len(set(recommended) & set(relevant)) / k


def recall_at_k(recommended, relevant, k: int = 10) -> float:
    recommended = recommended[:k]
    rel = set(relevant)
    if len(rel) == 0:
        return 0.0
    return len(set(recommended) & rel) / len(rel)


def ndcg_at_k(recommended, relevant, k: int = 10) -> float:
    recommended = recommended[:k]
    dcg = 0.0
    for idx, item in enumerate(recommended, start=1):
        if item in relevant:
            dcg += 1.0 / math.log2(idx + 1)
    ideal_hits = min(len(set(relevant)), k)
    if ideal_hits == 0:
        return 0.0
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg


def coverage_from_recs(all_recommendations: list[list], catalog_size: int, k: int = 10) -> float:
    recommended_items = set()
    for recs in all_recommendations:
        recommended_items.update(recs[:k])
    return len(recommended_items) / catalog_size if catalog_size > 0 else 0.0


def simple_diversity_at_k(recommended, k: int = 10) -> float:
    rec = recommended[:k]
    if not rec:
        return 0.0
    return len(set(rec)) / len(rec)


def serendipity_at_k(recommended, relevant, popular_set: set, k: int = 10) -> float:
    recommended = recommended[:k]
    rel = set(relevant)
    hits = [item for item in recommended if item in rel and item not in popular_set]
    return len(hits) / k if k > 0 else 0.0


def evaluate_context_aware_block(
    data_dir: Path,
    max_users: int = CONTEXT_EVAL_MAX_USERS,
    k: int = K,
    seed: int = RANDOM_SEED,
) -> dict | None:
    """Mirror 04_context_aware.ipynb: same split, weights, and recommend_context_aware logic."""
    req = ["orders.csv", "order_products__prior.csv"]
    if not all((data_dir / f).is_file() for f in req):
        return None

    TOP_N_PER_CONTEXT = 200
    W_HOUR, W_DOW, W_RECENCY = 0.4, 0.3, 0.3
    DOW_MAP = {
        0: "Saturday", 1: "Sunday", 2: "Monday", 3: "Tuesday",
        4: "Wednesday", 5: "Thursday", 6: "Friday",
    }

    orders = pd.read_csv(
        data_dir / "orders.csv",
        usecols=[
            "order_id", "user_id", "eval_set", "order_number",
            "order_dow", "order_hour_of_day", "days_since_prior_order",
        ],
        dtype={
            "order_id": "int32",
            "user_id": "int32",
            "eval_set": "category",
            "order_number": "int16",
            "order_dow": "int8",
            "order_hour_of_day": "int8",
            "days_since_prior_order": "float32",
        },
    )
    prior_items = pd.read_csv(
        data_dir / "order_products__prior.csv",
        usecols=["order_id", "product_id", "reordered"],
        dtype={"order_id": "int32", "product_id": "int32", "reordered": "int8"},
    )
    orders_prior = orders.loc[
        orders["eval_set"] == "prior",
        [
            "order_id", "user_id", "order_number", "order_dow",
            "order_hour_of_day", "days_since_prior_order",
        ],
    ].copy()
    del orders
    gc.collect()

    prior = prior_items.merge(orders_prior, on="order_id", how="inner", copy=False)
    del prior_items, orders_prior
    gc.collect()

    last_order = (
        prior.groupby("user_id", observed=True)["order_number"]
        .max()
        .rename("last_order_number")
    )
    prior = prior.merge(last_order, on="user_id", how="left", copy=False)

    train_df = prior.loc[prior["order_number"] < prior["last_order_number"]].copy()
    test_df = prior.loc[prior["order_number"] == prior["last_order_number"]].copy()
    valid_users = np.intersect1d(train_df["user_id"].unique(), test_df["user_id"].unique())
    train_df = train_df[train_df["user_id"].isin(valid_users)].copy()
    test_df = test_df[test_df["user_id"].isin(valid_users)].copy()
    del prior, last_order
    gc.collect()

    train_df["hour_bin"] = pd.cut(
        train_df["order_hour_of_day"],
        bins=[-1, 5, 11, 16, 20, 23],
        labels=["night", "morning", "afternoon", "evening", "night2"],
        ordered=False,
    ).astype("string").replace({"night2": "night"}).astype("category")

    train_df["recency_bin"] = pd.Series(
        np.select(
            [
                train_df["days_since_prior_order"].isna(),
                train_df["days_since_prior_order"] <= 7,
                train_df["days_since_prior_order"] <= 14,
            ],
            ["first_order", "frequent", "regular"],
            default="lapsed",
        ),
        index=train_df.index,
    ).astype("category")

    train_df["dow_label"] = train_df["order_dow"].map(DOW_MAP).astype("category")

    def build_context_table(df, context_col, top_n=TOP_N_PER_CONTEXT):
        grp = (
            df.groupby([context_col, "product_id"], observed=True)
            .agg(count=("product_id", "size"), reorder_rate=("reordered", "mean"))
            .reset_index()
        )
        grp["score"] = grp["reorder_rate"].astype("float32") * np.log1p(
            grp["count"].astype("float32")
        )
        grp = grp[[context_col, "product_id", "score"]]
        grp = grp.sort_values([context_col, "score"], ascending=[True, False])
        top_grp = grp.groupby(context_col, observed=True).head(top_n).copy()
        tables = {}
        for ctx, sub in top_grp.groupby(context_col, observed=True):
            tables[ctx] = dict(zip(sub["product_id"].astype(int), sub["score"].astype(float)))
        return tables

    hour_scores = build_context_table(train_df, "hour_bin")
    dow_scores = build_context_table(train_df, "dow_label")
    recency_scores = build_context_table(train_df, "recency_bin")

    def recommend_context_aware(hour, day_of_week, days_since_last_order, k=k):
        hour_label = (
            "morning"
            if 6 <= hour < 12
            else "afternoon"
            if 12 <= hour < 17
            else "evening"
            if 17 <= hour < 21
            else "night"
        )
        dow_label = DOW_MAP[int(day_of_week)]
        recency_label = (
            "first_order"
            if pd.isna(days_since_last_order)
            else "frequent"
            if days_since_last_order <= 7
            else "regular"
            if days_since_last_order <= 14
            else "lapsed"
        )
        combined = {}
        for table, weight, key in [
            (hour_scores, W_HOUR, hour_label),
            (dow_scores, W_DOW, dow_label),
            (recency_scores, W_RECENCY, recency_label),
        ]:
            for pid, score in table.get(key, {}).items():
                combined[pid] = combined.get(pid, 0.0) + weight * float(score)
        ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:k]
        return [int(pid) for pid, _ in ranked]

    ground_truth = (
        test_df.groupby("user_id")["product_id"].apply(lambda s: set(s.astype(int))).to_dict()
    )

    user_context = (
        test_df.groupby("user_id", observed=True)
        .first()[["order_hour_of_day", "order_dow", "days_since_prior_order"]]
    )

    rng = np.random.default_rng(seed)
    user_ids = [u for u in ground_truth if u in user_context.index]
    if len(user_ids) > max_users:
        user_ids = list(rng.choice(user_ids, size=max_users, replace=False))

    popular_set = set(train_df["product_id"].value_counts().head(20).index.astype(int))
    catalog_size = int(train_df["product_id"].nunique())

    precs, recs, ndcgs, divs, serends, all_reclists = [], [], [], [], [], []
    for uid in user_ids:
        row = user_context.loc[uid]
        rel = ground_truth[uid]
        rec = recommend_context_aware(
            hour=int(row["order_hour_of_day"]),
            day_of_week=int(row["order_dow"]),
            days_since_last_order=row["days_since_prior_order"],
            k=k,
        )
        precs.append(precision_at_k(rec, rel, k))
        recs.append(recall_at_k(rec, rel, k))
        ndcgs.append(ndcg_at_k(rec, rel, k))
        divs.append(simple_diversity_at_k(rec, k))
        serends.append(serendipity_at_k(rec, rel, popular_set, k))
        all_reclists.append(rec)

    coverage = coverage_from_recs(all_reclists, catalog_size, k)

    out = {
        "Approach": "Context-Aware (Heuristic)",
        "RMSE": np.nan,
        "MAE": np.nan,
        "Precision@10": float(np.mean(precs)),
        "Recall@10": float(np.mean(recs)),
        "NDCG": float(np.mean(ndcgs)),
        "Coverage": float(coverage),
        "Diversity": float(np.mean(divs)),
        "Serendipity": float(np.mean(serends)),
        "Context": "Temporal (day of week, hour of day, recency)",
        "_source": (
            f"computed in 05_evaluation.ipynb ({len(user_ids)} users, seed={seed}); "
            "matches 04_context_aware heuristic"
        ),
    }
    del train_df, test_df
    gc.collect()
    return out

In [ ]:
def row_to_series(d: dict) -> pd.Series:
    return pd.Series({k: v for k, v in d.items() if not str(k).startswith("_")})


ctx_eval = evaluate_context_aware_block(DATA_DIR, max_users=CONTEXT_EVAL_MAX_USERS, k=K, seed=RANDOM_SEED)

if ctx_eval is not None:
    context_metrics = ctx_eval
    print(
        "Context-aware metrics computed on disk (evaluated users ≤",
        CONTEXT_EVAL_MAX_USERS,
        "). Copy these values into FALLBACK_CONTEXT_METRICS in the constants cell "
        "if you need the table to render without CSVs.",
    )
else:
    context_metrics = FALLBACK_CONTEXT_METRICS
    print(
        "Instacart CSVs not found next to this notebook — context row shows FALLBACK_CONTEXT_METRICS "
        "(set those numbers after one successful local run). Paths checked:",
        ", ".join(str(DATA_DIR / f) for f in ["orders.csv", "order_products__prior.csv"]),
    )

comparison = pd.DataFrame([
    row_to_series(METRICS_NON_PERSONALIZED),
    row_to_series(METRICS_CF),
    row_to_series(METRICS_CONTENT_TFIDF),
    row_to_series(context_metrics),
])

print("\nCompiling group comparison table (K=10 ranking columns)…")
display(comparison.round(4))

print(
    "\nNotes:\n"
    "- Content-based is TF‑IDF over product text + aisle + department (03), not logistic regression.\n"
    "- Item-based CF row uses ranking metrics from 02; RMSE/MAE are omitted (no unified rating model).\n"
    "- Most Popular RMSE/MAE are from 01 (reorder likelihood vs. marginal product scores).\n"
    "- TF‑IDF RMSE/MAE are computed in 03 on binary relevance vs. cosine scores for the top‑K list."
)

Context-aware metrics computed on disk (evaluated users ≤ 500 ). Copy these values into FALLBACK_CONTEXT_METRICS in the constants cell if you need the table to render without CSVs.

Compiling group comparison table (K=10 ranking columns)…


,Approach,RMSE,MAE,Precision@10,Recall@10,NDCG,Coverage,Diversity,Serendipity,Context
0,Non-Personalized (Most Popular),0.7710,0.6359,0.0743,0.0733,0.1025,0.0004,0.3556,0.0000,Non-personalized popularity baseline
1,Collaborative Filtering (Item-Based),NaN,NaN,0.0378,0.0557,0.0594,0.2520,0.9700,0.0054,Personalized item similarity (top-1500 product...
2,Content-Based (TF-IDF),0.5185,0.5181,0.0025,0.0020,0.0018,0.0160,0.4183,0.8455,Product name + aisle + department
3,Context-Aware (Heuristic),NaN,NaN,0.0650,0.0614,0.0895,0.0002,1.0000,0.0028,"Temporal (day of week, hour of day, recency)"



Notes:
- Content-based is TF‑IDF over product text + aisle + department (03), not logistic regression.
- Item-based CF row uses ranking metrics from 02; RMSE/MAE are omitted (no unified rating model).
- Most Popular RMSE/MAE are from 01 (reorder likelihood vs. marginal product scores).
- TF‑IDF RMSE/MAE are computed in 03 on binary relevance vs. cosine scores for the top‑K list.
